## Testing different classification models

In [71]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

In [72]:
df = pd.read_csv("../data/processed/fraud_detection_dataset_preprocessed.csv")
df.head()

,Transaction_Amount,Account_Balance,Previous_Fraudulent_Activity,Card_Age,Transaction_Distance,Risk_Score,Fraud_Label,Transaction_Type_ATM Withdrawal,Transaction_Type_Bank Transfer,Transaction_Type_Online,...,Merchant_Category_Restaurants,Merchant_Category_Travel,Card_Type_Amex,Card_Type_Discover,Card_Type_Mastercard,Card_Type_Visa,Authentication_Method_Biometric,Authentication_Method_OTP,Authentication_Method_PIN,Authentication_Method_Password
0,39.79,93213.17,0,65,883.17,0.8494,0,False,False,False,...,False,True,True,False,False,False,True,False,False,False
1,1.19,75725.25,0,186,2203.36,0.0959,1,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,28.96,1588.96,0,226,1909.29,0.8400,1,False,False,True,...,True,False,False,False,False,True,True,False,False,False
3,254.32,76807.20,0,76,1311.86,0.7935,1,True,False,False,...,False,False,False,False,False,True,False,True,False,False
4,31.28,92354.66,1,140,966.98,0.3819,1,False,False,False,...,False,False,False,False,True,False,False,False,False,True


In [73]:
X = df.drop(columns=["Fraud_Label"])
y = df["Fraud_Label"]

In [74]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

In [75]:
from imblearn.over_sampling import SMOTE
sm = SMOTE()
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

In [76]:
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
X_train_res = pd.DataFrame(
    ss.fit_transform(X_train_res),
    columns=X.columns
)

X_test = pd.DataFrame(
    ss.transform(X_test),
    columns=X.columns
)

In [77]:
X_train_res.head()

,Transaction_Amount,Account_Balance,Previous_Fraudulent_Activity,Card_Age,Transaction_Distance,Risk_Score,Transaction_Type_ATM Withdrawal,Transaction_Type_Bank Transfer,Transaction_Type_Online,Transaction_Type_POS,...,Merchant_Category_Restaurants,Merchant_Category_Travel,Card_Type_Amex,Card_Type_Discover,Card_Type_Mastercard,Card_Type_Visa,Authentication_Method_Biometric,Authentication_Method_OTP,Authentication_Method_PIN,Authentication_Method_Password
0,-0.176536,1.688160,-0.282767,-0.400187,-1.005215,0.631202,1.535913,-0.653566,-0.655302,-0.658285,...,1.764286,-0.568517,-0.648756,1.533677,-0.658154,-0.658187,-0.651111,-0.652846,1.515474,-0.653239
1,-0.024014,0.388899,-0.282767,-0.822857,0.538283,-1.030022,-0.651079,-0.653566,-0.655302,1.519098,...,-0.566802,-0.568517,-0.648756,1.533677,-0.658154,-0.658187,1.535836,-0.652846,-0.659860,-0.653239
2,0.044964,0.300249,-0.282767,0.248914,1.457984,0.464487,1.535913,-0.653566,-0.655302,-0.658285,...,-0.566802,-0.568517,-0.648756,-0.652028,-0.658154,1.519326,-0.651111,1.531755,-0.659860,-0.653239
3,-1.034792,-0.168138,-0.282767,0.596107,-1.164426,1.408974,1.535913,-0.653566,-0.655302,-0.658285,...,-0.566802,1.758963,1.541411,-0.652028,-0.658154,-0.658187,-0.651111,-0.652846,-0.659860,1.530834
4,-0.927470,0.271170,3.536477,-0.883238,1.367872,1.242259,1.535913,-0.653566,-0.655302,-0.658285,...,-0.566802,-0.568517,-0.648756,-0.652028,-0.658154,1.519326,-0.651111,-0.652846,1.515474,-0.653239


In [78]:
y_train

8950     0
38421    0
19363    0
30157    1
14294    1
        ..
43723    0
32511    0
5192     1
12172    1
33003    0
Name: Fraud_Label, Length: 35000, dtype: int64

### Logistic Regression

In [79]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000, random_state=1)
lr.fit(X_train_res, y_train_res)
y_pred_lr = lr.predict(X_test)
confusion_matrix(y_test, y_pred_lr)

array([[9277,  864],
       [2360, 2499]], dtype=int64)

### Random Forest

In [80]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(max_depth=2, random_state=1)
rf.fit(X_train_res, y_train_res)
y_pred_rf = rf.predict(X_test)
confusion_matrix(y_test, y_pred_rf)


array([[10139,     2],
       [ 2570,  2289]], dtype=int64)

### Gradient Boosting

In [81]:
from sklearn.ensemble import GradientBoostingClassifier
gbc = GradientBoostingClassifier(n_estimators=100, learning_rate=1.0, max_depth=1, random_state=1)
gbc.fit(X_train_res, y_train_res)
y_pred_gbc = gbc.predict(X_test)
confusion_matrix(y_test, y_pred_gbc)

array([[10138,     3],
       [ 2570,  2289]], dtype=int64)

### Naive Bayes

In [82]:
from sklearn.naive_bayes import GaussianNB
nb = GaussianNB()
nb.fit(X_train_res, y_train_res)
y_pred_nb = nb.predict(X_test)
confusion_matrix(y_test, y_pred_nb)

array([[8341, 1800],
       [2370, 2489]], dtype=int64)